In [ ]:
import json
import os
from typing import Dict, List, Any, Union, Callable

def analyze_bug_predictions_with_multiple_judges(
    llm_judgments_paths: List[str],
    ground_truth_path: str,
    llm_predictions_path: str,
    combination_method: str = "union"
) -> Dict[str, Any]:
    """
    Analyze bug predictions from multiple LLM judgment files against ground truth.
    
    Args:
        llm_judgments_paths: List of paths to JSON files with LLM true/false judgments
        ground_truth_path: Path to ground truth bug paths JSON
        llm_predictions_path: Path to JSON with LLM predicted paths
        combination_method: How to combine judgments - "union" or "intersection"
        
    Returns:
        Dictionary with analysis metrics including accuracy by number of predicted paths
    """
    # Load ground truth bug paths
    with open(ground_truth_path, 'r') as f:
        ground_truth = json.load(f)
    
    # Load LLM predictions
    with open(llm_predictions_path, 'r') as f:
        llm_predictions = json.load(f)
    
    # Load all LLM judgments
    all_judgments = []
    for path in llm_judgments_paths:
        with open(path, 'r') as f:
            all_judgments.append(json.load(f))
    
    # Combine judgments based on method
    combined_judgments = {}
    
    for instance_id in llm_predictions:
        # Initialize with empty dict for this instance
        combined_judgments[instance_id] = {}
        
        # Get all predicted paths for this instance
        predicted_paths = llm_predictions[instance_id]
        
        for path in predicted_paths:
            # Check this path in all judgment files
            judgments_for_path = []
            
            for judgment in all_judgments:
                # If instance exists in this judgment and path is marked
                if (instance_id in judgment and 
                    path in judgment[instance_id]):
                    judgments_for_path.append(judgment[instance_id][path])
                else:
                    # Path not found in this judgment file, consider it False
                    judgments_for_path.append(False)
            
            # Apply combination method
            if combination_method == "union":
                # Path is valid if ANY judgment says True
                combined_judgments[instance_id][path] = any(judgments_for_path)
            elif combination_method == "intersection":
                # Path is valid if ALL judgments say True
                combined_judgments[instance_id][path] = all(judgments_for_path)
            elif combination_method == "voting":
                # Path is valid if majority (more than half) of judgments say True
                true_count = sum(1 for j in judgments_for_path if j is True)
                false_count = len(judgments_for_path) - true_count
                combined_judgments[instance_id][path] = true_count > false_count
    
    # Filter predictions based on combined judgments
    filtered_predictions = {}
    for instance_id, paths_dict in combined_judgments.items():
        # Get all paths that LLM predicted for this instance
        predicted_paths = llm_predictions[instance_id]
        
        # Filter to only keep paths where combined judgment is True
        filtered_paths = [
            path for path in predicted_paths 
            if path in paths_dict and paths_dict[path] is True
        ]
        
        filtered_predictions[instance_id] = filtered_paths
    
    # Calculate average paths per instance
    total_paths = sum(len(paths) for paths in filtered_predictions.values())
    total_instances = len(filtered_predictions)
    avg_paths_per_instance = total_paths / total_instances if total_instances > 0 else 0
    
    # Initialize counters for prediction count analysis
    max_path_count = 10  # Increase this if needed
    pred_count_instances = {i: 0 for i in range(1, max_path_count + 1)}
    pred_count_correct = {i: 0 for i in range(1, max_path_count + 1)}
    
    # Count instances by prediction count and track correctness
    for instance_id, filtered_paths in filtered_predictions.items():
        # Number of paths after filtering
        num_paths = len(filtered_paths)
        
        # Only analyze if prediction count is in our target range
        if 1 <= num_paths <= max_path_count:
            pred_count_instances[num_paths] += 1
            
            # Check if the ground truth path is in the filtered predictions
            if instance_id in ground_truth and ground_truth[instance_id] in filtered_paths:
                pred_count_correct[num_paths] += 1
    
    # Calculate accuracy for each prediction count
    pred_count_accuracy = {}
    for count in range(1, max_path_count + 1):
        accuracy = pred_count_correct[count] / pred_count_instances[count] if pred_count_instances[count] > 0 else 0
        pred_count_accuracy[count] = accuracy
    
    # Calculate overall metrics
    instances_with_zero_paths = sum(1 for paths in filtered_predictions.values() if len(paths) == 0)
    
    # Calculate regular accuracy across all instances
    correct_predictions = 0
    for instance_id, ground_truth_path in ground_truth.items():
        if instance_id in filtered_predictions:
            if ground_truth_path in filtered_predictions[instance_id]:
                correct_predictions += 1
    
    overall_accuracy = correct_predictions / total_instances if total_instances > 0 else 0
    
    # Return metrics
    return {
        "average_paths_per_instance": avg_paths_per_instance,
        "instances_with_zero_paths": instances_with_zero_paths,
        "instances_with_zero_paths_percentage": instances_with_zero_paths / total_instances if total_instances > 0 else 0,
        "overall_accuracy": overall_accuracy,
        "total_instances": total_instances,
        "total_paths": total_paths,
        "correct_predictions": correct_predictions,
        "pred_count_instances": pred_count_instances,
        "pred_count_correct": pred_count_correct,
        "pred_count_accuracy": pred_count_accuracy,
        "filtered_predictions": filtered_predictions,
        "combination_method": combination_method,
        "judgment_files_count": len(llm_judgments_paths)
    }

# Example usage
# Define file paths
llm_judgments_paths = [
    # "./localization_combination_results/verifications/model_results/mistral_largest/history.json",
    # "./localization_combination_results/verifications/model_results/gemini/gemini20_history.json",
    "./localization_combination_results/verifications/model_results/deepseek/history.json",
    "./localization_combination_results/verifications/model_results/chatgpt/history.json",
    "./localization_combination_results/verifications/model_results/claude/history.json"
]
ground_truth_path = "./ground_truth/bug_paths.json"
# llm_predictions_path = "./localization_combination_results/union/results/best_top_5_results.json"
llm_predictions_path = "./localization_combination_results/union/selected_results/union_top_3_results.json"

# Choose combination method: "union" or "intersection"
combination_method = "voting"
# combination_method = "intersection"
# combination_method = "union"

# Run the analysis
results = analyze_bug_predictions_with_multiple_judges(
    llm_judgments_paths=llm_judgments_paths,
    ground_truth_path=ground_truth_path,
    llm_predictions_path=llm_predictions_path,
    combination_method=combination_method
)

# Print basic results
print(f"Analysis Results using {combination_method.upper()} of {results['judgment_files_count']} judgment files:")
print(f"Average paths per instance: {results['average_paths_per_instance']:.2f}")
print(f"Total paths across all instances: {results['total_paths']}")
print(f"Instances with zero paths: {results['instances_with_zero_paths']} ({results['instances_with_zero_paths_percentage']:.2%})")
print(f"Overall accuracy: {results['overall_accuracy']:.2%}")
print(f"Total instances analyzed: {results['total_instances']}")
print(f"Correct predictions: {results['correct_predictions']}")

# Print accuracy by prediction count
print("\nAccuracy by Number of Predicted Paths:")
for count in range(1, 11):  # Adjust range as needed
    instances = results['pred_count_instances'][count]
    correct = results['pred_count_correct'][count]
    accuracy = results['pred_count_accuracy'][count]
    if instances > 0:
        print(f"Predictions with {count} path(s): {correct}/{instances} correct ({accuracy:.2%})")

# with open('./localization_candidates/union_top2union_gptclaudedeepseek.json', 'w') as f:
#     json.dump(results['filtered_predictions'], f, indent=4)

In [ ]:
# import json
# import os
# from typing import Dict, List, Any

# def analyze_bug_predictions(
#     llm_judgments_path: str,
#     ground_truth_path: str,
#     llm_predictions_path: str
# ) -> Dict[str, Any]:
#     """
#     Analyze bug predictions from LLM judgments against ground truth.
    
#     Args:
#         llm_judgments_path: Path to JSON with LLM true/false judgments
#         ground_truth_path: Path to ground truth bug paths JSON
#         llm_predictions_path: Path to JSON with LLM predicted paths
        
#     Returns:
#         Dictionary with analysis metrics
#     """
#     # Load LLM judgments (true/false decisions)
#     with open(llm_judgments_path, 'r') as f:
#         llm_judgments = json.load(f)
    
#     # Load ground truth bug paths
#     with open(ground_truth_path, 'r') as f:
#         ground_truth = json.load(f)
    
#     # Load LLM predictions
#     with open(llm_predictions_path, 'r') as f:
#         llm_predictions = json.load(f)
    
#     # Filter predictions based on LLM judgments (keep only paths marked as True)
#     filtered_predictions = {}
#     for instance_id, paths_dict in llm_judgments.items():
#         if instance_id in llm_predictions:
#             # Get all paths that LLM predicted for this instance
#             predicted_paths = llm_predictions[instance_id]
            
#             # Filter to only keep paths where LLM judgment is True
#             filtered_paths = [
#                 path for path in predicted_paths 
#                 if path in paths_dict and paths_dict[path] is True
#             ]
            
#             filtered_predictions[instance_id] = filtered_paths
    
#     # Calculate metrics
#     total_instances = len(filtered_predictions)
#     total_paths = sum(len(paths) for paths in filtered_predictions.values())
#     instances_with_zero_paths = sum(1 for paths in filtered_predictions.values() if len(paths) == 0)
    
#     # Calculate average number of paths per instance
#     avg_paths = total_paths / total_instances if total_instances > 0 else 0
    
#     # Calculate accuracy: instances where ground truth path is in the filtered predictions
#     correct_predictions = 0
#     for instance_id, ground_truth_path in ground_truth.items():
#         if instance_id in filtered_predictions:
#             if ground_truth_path in filtered_predictions[instance_id]:
#                 correct_predictions += 1
    
#     accuracy = correct_predictions / total_instances if total_instances > 0 else 0
    
#     # Return metrics
#     return {
#         "average_paths": avg_paths,
#         "instances_with_zero_paths": instances_with_zero_paths,
#         "instances_with_zero_paths_percentage": instances_with_zero_paths / total_instances if total_instances > 0 else 0,
#         "accuracy": accuracy,
#         "total_instances": total_instances,
#         "correct_predictions": correct_predictions, 
#         "filtered_predictions": filtered_predictions
#     }


# # Define file paths
# llm_judgments_path = "./localization_combination_results/verifications/model_results/mistral/history.json"
# # llm_judgments_path = "./localization_combination_results/verifications/model_results/deepseek/history.json"
# ground_truth_path = "./ground_truth/bug_paths.json"
# llm_predictions_path = "./localization_combination_results/union/results/best_top_2_results.json"
# # llm_predictions_path = "./localization/hierarchy_0/claude/20250327_174452/predictions.json"

# # Run analysis
# results = analyze_bug_predictions(
#     llm_judgments_path,
#     ground_truth_path,
#     llm_predictions_path
# )

# # Print results
# print(f"Analysis Results:")
# print(f"Average paths per instance: {results['average_paths']:.2f}")
# print(f"Instances with zero paths: {results['instances_with_zero_paths']} ({results['instances_with_zero_paths_percentage']:.2%})")
# print(f"Accuracy: {results['accuracy']:.2%}")
# print(f"Total instances analyzed: {results['total_instances']}")
# print(f"Correct predictions: {results['correct_predictions']}")

In [ ]:
# import json
# import os
# from typing import Dict, List, Any, Union, Callable

# def analyze_bug_predictions_with_multiple_judges(
#     llm_judgments_paths: List[str],
#     ground_truth_path: str,
#     llm_predictions_path: str,
#     combination_method: str = "union"
# ) -> Dict[str, Any]:
#     """
#     Analyze bug predictions from multiple LLM judgment files against ground truth.
    
#     Args:
#         llm_judgments_paths: List of paths to JSON files with LLM true/false judgments
#         ground_truth_path: Path to ground truth bug paths JSON
#         llm_predictions_path: Path to JSON with LLM predicted paths
#         combination_method: How to combine judgments - "union" or "intersection"
        
#     Returns:
#         Dictionary with analysis metrics including accuracy by number of predicted paths
#     """
#     # Load ground truth bug paths
#     with open(ground_truth_path, 'r') as f:
#         ground_truth = json.load(f)
    
#     # Load LLM predictions
#     with open(llm_predictions_path, 'r') as f:
#         llm_predictions = json.load(f)
    
#     # Load all LLM judgments
#     all_judgments = []
#     for path in llm_judgments_paths:
#         with open(path, 'r') as f:
#             all_judgments.append(json.load(f))
    
#     # Combine judgments based on method
#     combined_judgments = {}
    
#     for instance_id in llm_predictions:
#         # Initialize with empty dict for this instance
#         combined_judgments[instance_id] = {}
        
#         # Get all predicted paths for this instance
#         predicted_paths = llm_predictions[instance_id]
        
#         for path in predicted_paths:
#             # Check this path in all judgment files
#             judgments_for_path = []
            
#             for judgment in all_judgments:
#                 # If instance exists in this judgment and path is marked
#                 if (instance_id in judgment and 
#                     path in judgment[instance_id]):
#                     judgments_for_path.append(judgment[instance_id][path])
#                 else:
#                     # Path not found in this judgment file, consider it False
#                     judgments_for_path.append(False)
            
#             # Apply combination method
#             if combination_method == "union":
#                 # Path is valid if ANY judgment says True
#                 combined_judgments[instance_id][path] = any(judgments_for_path)
#             else:  # intersection
#                 # Path is valid if ALL judgments say True
#                 combined_judgments[instance_id][path] = all(judgments_for_path)
    
#     # Filter predictions based on combined judgments
#     filtered_predictions = {}
#     for instance_id, paths_dict in combined_judgments.items():
#         # Get all paths that LLM predicted for this instance
#         predicted_paths = llm_predictions[instance_id]
        
#         # Filter to only keep paths where combined judgment is True
#         filtered_paths = [
#             path for path in predicted_paths 
#             if path in paths_dict and paths_dict[path] is True
#         ]
        
#         filtered_predictions[instance_id] = filtered_paths
    
#     # Initialize counters for prediction count analysis
#     max_path_count = 10  # Increase this if needed
#     pred_count_instances = {i: 0 for i in range(1, max_path_count + 1)}
#     pred_count_correct = {i: 0 for i in range(1, max_path_count + 1)}
    
#     # Count instances by prediction count and track correctness
#     for instance_id, filtered_paths in filtered_predictions.items():
#         # Number of paths after filtering
#         num_paths = len(filtered_paths)
        
#         # Only analyze if prediction count is in our target range
#         if 1 <= num_paths <= max_path_count:
#             pred_count_instances[num_paths] += 1
            
#             # Check if the ground truth path is in the filtered predictions
#             if instance_id in ground_truth and ground_truth[instance_id] in filtered_paths:
#                 pred_count_correct[num_paths] += 1
    
#     # Calculate accuracy for each prediction count
#     pred_count_accuracy = {}
#     for count in range(1, max_path_count + 1):
#         accuracy = pred_count_correct[count] / pred_count_instances[count] if pred_count_instances[count] > 0 else 0
#         pred_count_accuracy[count] = accuracy
    
#     # Calculate overall metrics
#     total_instances = len(filtered_predictions)
#     instances_with_zero_paths = sum(1 for paths in filtered_predictions.values() if len(paths) == 0)
    
#     # Calculate regular accuracy across all instances
#     correct_predictions = 0
#     for instance_id, ground_truth_path in ground_truth.items():
#         if instance_id in filtered_predictions:
#             if ground_truth_path in filtered_predictions[instance_id]:
#                 correct_predictions += 1
    
#     overall_accuracy = correct_predictions / total_instances if total_instances > 0 else 0
    
#     # Return metrics
#     return {
#         "instances_with_zero_paths": instances_with_zero_paths,
#         "instances_with_zero_paths_percentage": instances_with_zero_paths / total_instances if total_instances > 0 else 0,
#         "overall_accuracy": overall_accuracy,
#         "total_instances": total_instances,
#         "correct_predictions": correct_predictions,
#         "pred_count_instances": pred_count_instances,
#         "pred_count_correct": pred_count_correct,
#         "pred_count_accuracy": pred_count_accuracy,
#         "filtered_predictions": filtered_predictions,
#         "combination_method": combination_method,
#         "judgment_files_count": len(llm_judgments_paths)
#     }

# # Example usage
# # Define file paths
# llm_judgments_paths = [
#     "./localization_combination_results/verifications/model_results/mistral_largest/history.json",
#     "./localization_combination_results/verifications/model_results/claude/history.json",
#     "./localization_combination_results/verifications/model_results/chatgpt/history.json",
#     "./localization_combination_results/verifications/model_results/deepseek/history.json"
# ]
# ground_truth_path = "./ground_truth/bug_paths.json"
# llm_predictions_path = "./localization_combination_results/union/results/best_top_2_results.json"

# # Choose combination method: "union" or "intersection"
# combination_method = "union"

# # Run the analysis
# results = analyze_bug_predictions_with_multiple_judges(
#     llm_judgments_paths=llm_judgments_paths,
#     ground_truth_path=ground_truth_path,
#     llm_predictions_path=llm_predictions_path,
#     combination_method=combination_method
# )

# # Print basic results
# print(f"Analysis Results using {combination_method.upper()} of {results['judgment_files_count']} judgment files:")
# print(f"Instances with zero paths: {results['instances_with_zero_paths']} ({results['instances_with_zero_paths_percentage']:.2%})")
# print(f"Overall accuracy: {results['overall_accuracy']:.2%}")
# print(f"Total instances analyzed: {results['total_instances']}")
# print(f"Correct predictions: {results['correct_predictions']}")

# # Print accuracy by prediction count
# print("\nAccuracy by Number of Predicted Paths:")
# for count in range(1, 11):  # Adjust range as needed
#     instances = results['pred_count_instances'][count]
#     correct = results['pred_count_correct'][count]
#     accuracy = results['pred_count_accuracy'][count]
#     if instances > 0:
#         print(f"Predictions with {count} path(s): {correct}/{instances} correct ({accuracy:.2%})")

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Assuming results is your dictionary containing filtered_predictions
# # results = {...}  # Your actual data here
# filtered_predictions = results['filtered_predictions']

# # Calculate all path lengths
# lengths = []
# max_length, min_length = -1, float('inf')

# for key in filtered_predictions.keys():
#     paths = filtered_predictions[key]
#     paths_len = len(paths)
#     lengths.append(paths_len)
    
#     if paths_len > max_length:
#         max_length = paths_len
#     if paths_len < min_length:
#         min_length = paths_len

# print(f"Max length: {max_length}")
# print(f"Min length: {min_length}")

# # Create a histogram of the lengths
# plt.figure(figsize=(10, 6))
# bins = range(min_length, max_length + 2)  # +2 to include the max value in its own bin

# # Plot the histogram
# plt.hist(lengths, bins=bins, alpha=0.7, color='skyblue', edgecolor='black')

# # Add labels and title
# plt.xlabel('Path Length', fontsize=12)
# plt.ylabel('Frequency', fontsize=12)
# plt.title('Distribution of Path Lengths', fontsize=14)

# # Set x-tick positions to be at the center of each bin
# plt.xticks(np.arange(min_length, max_length + 1, 1))

# # Add grid for better readability
# plt.grid(axis='y', alpha=0.3)

# # Add text with summary statistics
# plt.text(0.7, 0.9, f'Min Length: {min_length}\nMax Length: {max_length}', 
#          transform=plt.gca().transAxes, fontsize=10,
#          bbox=dict(facecolor='white', alpha=0.8))

# plt.tight_layout()
# plt.show()

In [ ]:
# import json
# import os
# from typing import Dict, List, Any

# def analyze_bug_predictions(
#     llm_judgments_path: str,
#     ground_truth_path: str,
#     llm_predictions_path: str,
#     complementary_predictions_path: str = None
# ) -> Dict[str, Any]:
#     """
#     Analyze bug predictions from LLM judgments against ground truth.
    
#     Args:
#         llm_judgments_path: Path to JSON with LLM true/false judgments
#         ground_truth_path: Path to ground truth bug paths JSON
#         llm_predictions_path: Path to JSON with LLM predicted paths
#         complementary_predictions_path: Path to complementary predictions JSON (fallback)
        
#     Returns:
#         Dictionary with analysis metrics
#     """
#     # Load LLM judgments (true/false decisions)
#     with open(llm_judgments_path, 'r') as f:
#         llm_judgments = json.load(f)
    
#     # Load ground truth bug paths
#     with open(ground_truth_path, 'r') as f:
#         ground_truth = json.load(f)
    
#     # Load LLM predictions
#     with open(llm_predictions_path, 'r') as f:
#         llm_predictions = json.load(f)
    
#     # Load complementary predictions if provided
#     complementary_predictions = {}
#     if complementary_predictions_path:
#         with open(complementary_predictions_path, 'r') as f:
#             complementary_predictions = json.load(f)
    
#     # Filter predictions based on LLM judgments (keep only paths marked as True)
#     filtered_predictions = {}
#     fallback_used_count = 0
    
#     for instance_id, paths_dict in llm_judgments.items():
#         if instance_id in llm_predictions:
#             # Get all paths that LLM predicted for this instance
#             predicted_paths = llm_predictions[instance_id]
            
#             # Filter to only keep paths where LLM judgment is True
#             filtered_paths = [
#                 path for path in predicted_paths 
#                 if path in paths_dict and paths_dict[path] is True
#             ]
            
#             # If we have zero paths after filtering and complementary predictions are available,
#             # use the complementary predictions as fallback
#             if len(filtered_paths) == 0 and complementary_predictions and instance_id in complementary_predictions:
#                 print(instance_id)
#                 filtered_paths = complementary_predictions[instance_id]
#                 fallback_used_count += 1
            
#             filtered_predictions[instance_id] = filtered_paths
    
#     # Calculate metrics
#     total_instances = len(filtered_predictions)
#     total_paths = sum(len(paths) for paths in filtered_predictions.values())
#     instances_with_zero_paths = sum(1 for paths in filtered_predictions.values() if len(paths) == 0)
    
#     # Calculate average number of paths per instance
#     avg_paths = total_paths / total_instances if total_instances > 0 else 0
    
#     # Calculate accuracy: instances where ground truth path is in the filtered predictions
#     correct_predictions = 0
#     for instance_id, ground_truth_path in ground_truth.items():
#         if instance_id in filtered_predictions:
#             if ground_truth_path in filtered_predictions[instance_id]:
#                 correct_predictions += 1
    
#     accuracy = correct_predictions / total_instances if total_instances > 0 else 0
    
#     # Return metrics
#     return {
#         "average_paths": avg_paths,
#         "instances_with_zero_paths": instances_with_zero_paths,
#         "instances_with_zero_paths_percentage": instances_with_zero_paths / total_instances if total_instances > 0 else 0,
#         "accuracy": accuracy,
#         "total_instances": total_instances,
#         "correct_predictions": correct_predictions,
#         "fallback_used_count": fallback_used_count,
#         "fallback_used_percentage": fallback_used_count / total_instances if total_instances > 0 else 0
#     }

# # Define file paths
# llm_judgments_path = "./localization_combination_results/verifications/model_results/deepseek/history.json"
# ground_truth_path = "./ground_truth/bug_paths.json"
# llm_predictions_path = "./localization_combination_results/union/results/best_top_2_results.json"
# complementary_predictions_path = "./localization_combination_results/intersect/results/best_top_2_results.json"

# # Run analysis
# results = analyze_bug_predictions(
#     llm_judgments_path,
#     ground_truth_path,
#     llm_predictions_path,
#     complementary_predictions_path
# )

# # Print results
# print(f"Analysis Results:")
# print(f"Average paths per instance: {results['average_paths']:.2f}")
# print(f"Instances with zero paths: {results['instances_with_zero_paths']} ({results['instances_with_zero_paths_percentage']:.2%})")
# print(f"Instances using fallback predictions: {results['fallback_used_count']} ({results['fallback_used_percentage']:.2%})")
# print(f"Accuracy: {results['accuracy']:.2%}")
# print(f"Total instances analyzed: {results['total_instances']}")
# print(f"Correct predictions: {results['correct_predictions']}")